# Stage 1 — Continued Pre-Training (CPT)

**Goal:** Align `google/gemma-3-1b-it` vocabulary and internal representations with natural Swahili by training on two large monolingual Swahili corpora.

This phase addresses the core failure observed in Stage 0: the base model has no Swahili vocabulary alignment, causing it to drift to other languages (Finnish) during generation.

## What CPT does
- Updates the embedding weights so common Swahili morphemes cluster together in vector space
- Reduces sub-word token fragmentation (e.g. *tutawasiliana* broken into 5+ arbitrary tokens → 2–3 meaningful ones)
- Teaches the model natural Swahili sentence flow before any translation supervision

## Datasets
| Dataset | Rows | Tokens | Content |
|---------|------|--------|---------|
| `AlexLeoTz/jamiiforums-2b` | 979,304 | ~1.26B | Colloquial Swahili forum discussions, slang, politics |
| `Alfaxad2002/inkuba-mono` (sw) | ~1M | ~60.8M | Clean monolingual Swahili text continuation |

## Training approach
- **QLoRA**: 4-bit quantized base + LoRA adapters (fits in 15.6 GB T4 VRAM)
- **Causal LM objective**: next-token prediction on raw Swahili text (no instruction formatting)
- **Subset training**: 80K samples from each dataset (~160K total) — enough for vocabulary alignment within Kaggle's session limit
- **Spec hyperparameters**: LR 2×10⁻⁵, cosine decay, context 1024 tokens (reduced from 8192 for T4 VRAM)

> **Kaggle setup:** GPU T4 x2 (use cuda:0 only) · Internet ON · HF_TOKEN secret set

## 1. Install dependencies

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES set to:", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
%%capture
!pip install -q transformers accelerate datasets peft trl bitsandbytes sacrebleu huggingface_hub

## 2. Environment check

In [ ]:
import torch
import transformers
import peft
import trl
import platform

print("=" * 55)
print("ENVIRONMENT CHECK")
print("=" * 55)
print(f"Python          : {platform.python_version()}")
print(f"PyTorch         : {torch.__version__}")
print(f"Transformers    : {transformers.__version__}")
print(f"PEFT            : {peft.__version__}")
print(f"TRL             : {trl.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}           : {torch.cuda.get_device_name(i)} ({mem:.1f} GB)")

# Pin to first GPU — same strategy as Stage 0
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"\nActive device   : {DEVICE}")

## 3. Authenticate with Hugging Face

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication successful.")
except Exception as e:
    print(f"Auth failed: {e}")

## 4. Configuration — all hyperparameters in one place

In [ ]:
import os

# ── Model ────────────────────────────────────────────────────
MODEL_ID        = "google/gemma-3-1b-it"
OUTPUT_DIR      = "/kaggle/working/stage1_cpt_adapter"
FINAL_MODEL_DIR = "/kaggle/working/stage1_cpt_merged"

# ── Data ─────────────────────────────────────────────────────
JAMII_SAMPLES   = 20_000    # rows from jamiiforums-2b
INKUBA_SAMPLES  = 20_000    # rows from inkuba-mono
MAX_SEQ_LEN     = 1024
TEXT_COLUMN     = "text"

# ── Training ──────────────────────────────────────────────────
LEARNING_RATE   = 2e-5
LR_SCHEDULER    = "cosine"
NUM_EPOCHS      = 1
BATCH_SIZE      = 2
GRAD_ACCUM      = 8         # effective batch = 16
WARMUP_RATIO    = 0.05
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 1.0
SAVE_STEPS      = 500
LOGGING_STEPS   = 50

# ── LoRA ─────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration set.")
print(f"  Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total training rows  : {JAMII_SAMPLES + INKUBA_SAMPLES:,}")
print(f"  Max sequence length  : {MAX_SEQ_LEN}")
print(f"  LoRA rank            : {LORA_R}")
print(f"  Estimated steps      : {int(JAMII_SAMPLES * 0.95 + INKUBA_SAMPLES * 0.95) // (BATCH_SIZE * GRAD_ACCUM):,}")


## 5. Load tokenizer

In [ ]:
from transformers import AutoTokenizer

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Gemma uses EOS as pad token — required for batch training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocab size      : {tokenizer.vocab_size:,}")
print(f"Pad token       : {tokenizer.pad_token!r}")
print(f"EOS token       : {tokenizer.eos_token!r}")

## 6. Pre-CPT tokenizer fragmentation baseline

Records the avg tokens/word on Swahili text **before** CPT. We compare this again after training to confirm vocabulary alignment improved.

In [ ]:
# Canonical Swahili test sentences covering morphological complexity
SWAHILI_AUDIT_SENTENCES = [
    "Tutawasiliana nawe kesho asubuhi mapema iwezekanavyo.",
    "Wanaopigana vita hawatarudisha amani katika nchi yetu.",
    "Utekelezaji wa sera hii utachukua muda mrefu sana.",
    "Maendeleo ya teknolojia yamesaidia watu wengi duniani.",
    "Uhusiano kati ya nchi mbili unategemea ushirikiano wa kweli.",
    "Wanafunzi wanaosoma kwa bidii watafanikiwa maishani.",
    "Serikali imetekeleza miradi mingi ya kuendeleza miundombinu.",
    "Waganga wanaofanya kazi hospitalini wanajitahidi sana.",
    "Uchumi wa taifa unakua polepole licha ya changamoto nyingi.",
    "Mazingira yanayotuzunguka yanahitaji kulindwa kwa makini.",
]

def fragmentation_audit(sentences, label=""):
    total_tokens, total_words = 0, 0
    details = []
    for sent in sentences:
        words  = sent.split()
        tokens = tokenizer.encode(sent, add_special_tokens=False)
        ratio  = len(tokens) / len(words)
        total_tokens += len(tokens)
        total_words  += len(words)
        details.append((ratio, sent[:55]))
    avg = total_tokens / total_words

    print(f"\n{'='*55}")
    print(f"TOKENIZER FRAGMENTATION AUDIT {label}")
    print(f"{'='*55}")
    print(f"Sentences       : {len(sentences)}")
    print(f"Total words     : {total_words}")
    print(f"Total tokens    : {total_tokens}")
    print(f"Avg tokens/word : {avg:.3f}")
    print("\nPer-sentence breakdown:")
    for ratio, snip in sorted(details, reverse=True):
        print(f"  {ratio:.2f}  {snip}")

    # Single-word deep-dive
    test_words = ["tutawasiliana", "wanaopigana", "utekelezaji", "maendeleo", "uhusiano"]
    print("\nSingle-word token breakdown:")
    for w in test_words:
        toks = tokenizer.convert_ids_to_tokens(
            tokenizer.encode(w, add_special_tokens=False)
        )
        print(f"  {w:22s} → {toks}  ({len(toks)} tokens)")

    return avg

pre_cpt_ratio = fragmentation_audit(SWAHILI_AUDIT_SENTENCES, label="(PRE-CPT)")

## 7. Load and prepare datasets

In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset

# ── Dataset 1: JamiiForums-2b ────────────────────────────────
print("Loading AlexLeoTz/jamiiforums-2b (train split)...")
jamii = load_dataset(
    "AlexLeoTz/jamiiforums-2b",
    split=f"train[:{JAMII_SAMPLES}]"
)
print(f"JamiiForums rows loaded : {len(jamii):,}")
print(f"Columns                 : {jamii.column_names}")
print(f"Sample text             : {str(jamii[0])[:200]}")

In [ ]:
# ── Dataset 2: Inkuba-Mono Swahili ───────────────────────────
# Two options — try the official lelapa HF repo first, fall back to Alfaxad mirror

print("Loading Inkuba-Mono Swahili...")

try:
    # Option A: Official lelapa repo (requires accepting conditions on HF)
    inkuba = load_dataset(
        "lelapa/Inkuba-Mono",
        "sw",
        split=f"train[:{INKUBA_SAMPLES}]"
    )
    print("Loaded from lelapa/Inkuba-Mono")

except Exception as e1:
    print(f"lelapa/Inkuba-Mono failed ({e1}), trying Alfaxad mirror...")
    try:
        # Option B: Alfaxad mirror (Swahili-only extract, no conditions)
        inkuba = load_dataset(
            "Alfaxad/Inkuba-Mono-Swahili",
            split=f"train[:{INKUBA_SAMPLES}]"
        )
        print("Loaded from Alfaxad/Inkuba-Mono-Swahili")

    except Exception as e2:
        print(f"HF mirror also failed ({e2})")
        print("Falling back to Kaggle dataset — add it via Add Data → Datasets → 'Inkuba-Mono-Swahili'")
        import pandas as pd
        inkuba_df = pd.read_parquet("/kaggle/input/inkuba-mono-swahili/data.parquet")
        inkuba = Dataset.from_pandas(inkuba_df.head(INKUBA_SAMPLES))
        print(f"Loaded from Kaggle input: {len(inkuba):,} rows")

print(f"Inkuba rows loaded      : {len(inkuba):,}")
print(f"Columns                 : {inkuba.column_names}")
print(f"Sample                  : {str(inkuba[0])[:200]}")

In [ ]:
from datasets import concatenate_datasets, Value, Features

def normalise_to_text(dataset):
    cols = dataset.column_names
    if "text" in cols:
        ds = dataset.select_columns(["text"])
    else:
        for candidate in ["content", "body", "post", "message", "sentence"]:
            if candidate in cols:
                ds = dataset.rename_column(candidate, "text").select_columns(["text"])
                break
        else:
            for col in cols:
                if "string" in str(dataset.features[col]).lower():
                    print(f"Using column '{col}' as text.")
                    ds = dataset.rename_column(col, "text").select_columns(["text"])
                    break
            else:
                raise ValueError(f"No suitable text column found. Columns: {cols}")

    # Cast to consistent string type to allow concatenation
    ds = ds.cast(Features({"text": Value("string")}))
    return ds

jamii_clean  = normalise_to_text(jamii)
inkuba_clean = normalise_to_text(inkuba)

combined = concatenate_datasets([jamii_clean, inkuba_clean]).shuffle(seed=42)
combined = combined.filter(lambda x: x["text"] and len(x["text"].strip()) > 30)

print(f"Combined dataset size   : {len(combined):,} rows")
print(f"Sample after normalise  : {combined[0]['text'][:150]}")

In [ ]:
# ── Tokenise ──────────────────────────────────────────────────
def tokenise(batch):
    """
    Tokenise raw Swahili text for causal language modelling.
    Do NOT set labels here — DataCollatorForLanguageModeling
    generates them correctly at batch time, masking padding to -100.
    """
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        return_attention_mask=True,
    )
    return out

print("Tokenising dataset...")
tokenised = combined.map(
    tokenise,
    batched=True,
    batch_size=1000,
    remove_columns=["text"],
    desc="Tokenising",
)

# Train only — no eval split needed since eval_strategy=no
split = tokenised.train_test_split(test_size=0.02, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]   # kept for Trainer signature but never used

print(f"Train rows              : {len(train_dataset):,}")
print(f"Eval rows               : {len(eval_dataset):,} (unused)")


## 8. Load base model with 4-bit quantization (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"" : 0},
    dtype=torch.bfloat16,
    attn_implementation="eager",
)

# use_reentrant=False silences the PyTorch 2.9 deprecation warning
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded. Parameters: {total_params:.2f}B")

allocated = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after model load   : {allocated:.2f} GB")


## 9. Apply LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Prepare quantized model for k-bit training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    inference_mode=False,
)

model = get_peft_model(model, lora_config)

# Parameter breakdown
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters    : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"Total parameters        : {total:,}")
print(f"Frozen parameters       : {total - trainable:,}")

allocated = torch.cuda.memory_allocated() / 1e9
print(f"VRAM after LoRA setup   : {allocated:.2f} GB")

## 10. Training configuration

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,          # fixed: was warmup_steps=WARMUP_RATIO (float bug)
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    fp16=False,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    eval_strategy="no",                 # no eval passes — saves ~7300s per checkpoint
    load_best_model_at_end=False,       # must be False when eval_strategy=no
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=2,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    dataloader_pin_memory=False,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

steps = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print("Training configuration ready.")
print(f"  Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Total steps         : {steps:,}")
print(f"  Est. training time  : {steps * 6 / 3600:.1f} hours (no eval overhead)")
print(f"  Optimizer           : paged_adamw_8bit")
print(f"  Scheduler           : {LR_SCHEDULER}")
print(f"  Learning rate       : {LEARNING_RATE}")


## 11. Train

In [ ]:
from transformers import Trainer
import time

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    # eval_dataset omitted — eval_strategy=no
    data_collator=data_collator,
)

steps = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print("Starting CPT training...")
print(f"Total steps             : {steps:,}")
print(f"Estimated time          : ~{steps * 6 / 3600:.1f} hours on T4")
print(f"Save checkpoint every   : {SAVE_STEPS} steps")
print()

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print(f"
Training complete in {elapsed/3600:.2f} hours.")
print(f"Final train loss        : {train_result.training_loss:.4f}")


## 12. Save the LoRA adapter

In [ ]:
import json

# Save adapter weights
adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

# Save training metrics
metrics = {
    "stage"          : 1,
    "phase"          : "CPT",
    "base_model"     : MODEL_ID,
    "train_loss"     : round(train_result.training_loss, 4),
    "train_samples"  : len(train_dataset),
    "eval_samples"   : len(eval_dataset),
    "epochs"         : NUM_EPOCHS,
    "lora_r"         : LORA_R,
    "lora_alpha"     : LORA_ALPHA,
    "learning_rate"  : LEARNING_RATE,
    "max_seq_len"    : MAX_SEQ_LEN,
    "jamii_samples"  : JAMII_SAMPLES,
    "inkuba_samples" : INKUBA_SAMPLES,
    "elapsed_hours"  : round(elapsed/3600, 2),
}

with open(f"{OUTPUT_DIR}/stage1_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"LoRA adapter saved to   : {adapter_path}")
print(f"Metrics saved to        : {OUTPUT_DIR}/stage1_metrics.json")

## 13. Post-CPT fragmentation audit

Compare tokenizer fragmentation before and after CPT. Note: the tokenizer vocabulary itself hasn't changed (LoRA doesn't modify it), but this confirms our embedding shifts are reflected in how the model *processes* Swahili morphemes at generation time.

In [ ]:
post_cpt_ratio = fragmentation_audit(SWAHILI_AUDIT_SENTENCES, label="(POST-CPT)")

print("\n" + "=" * 55)
print("FRAGMENTATION COMPARISON")
print("=" * 55)
print(f"Pre-CPT  avg tokens/word : {pre_cpt_ratio:.3f}")
print(f"Post-CPT avg tokens/word : {post_cpt_ratio:.3f}")
delta = pre_cpt_ratio - post_cpt_ratio
if delta > 0:
    print(f"Improvement              : -{delta:.3f} tokens/word ({100*delta/pre_cpt_ratio:.1f}% reduction)")
else:
    print("Note: tokenizer vocab unchanged — fragmentation change reflects embedding space only.")
    print("The key metric is translation quality improvement in cell 14.")

## 14. Post-CPT smoke test — qualitative translation check

In [ ]:
from peft import PeftModel

# Load the CPT model in inference mode
model.eval()

GENERATION_CONFIG = {
    "temperature"        : 0.3,
    "top_p"              : 0.95,
    "top_k"              : 64,
    "max_new_tokens"     : 128,
    "repetition_penalty" : 1.1,
    "do_sample"          : True,
}

def build_prompt(english_sentence: str) -> str:
    """Bilingual prompt — English input, Swahili-only output constraint."""
    return (
        "<start_of_turn>user\n"
        f"Translate the following English sentence to Swahili.\n"
        f"English: {english_sentence}\n"
        f"Swahili:"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )

def translate(sentence: str) -> str:
    prompt  = build_prompt(sentence)
    inputs  = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            **GENERATION_CONFIG,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Diverse test sentences — checks different domains
TEST_SENTENCES = [
    ("The students are learning at school today.",
     "Wanafunzi wanasoma shuleni leo."),
    ("She went to the market to buy vegetables and fruit.",
     "Alienda sokoni kununua mboga na matunda."),
    ("The government announced new policies for healthcare.",
     "Serikali ilitangaza sera mpya za huduma za afya."),
    ("Climate change is affecting agriculture across Africa.",
     "Mabadiliko ya tabianchi yanaathiri kilimo katika Afrika yote."),
    ("The beautiful chairs fell on the ground.",
     "Viti vizuri vilianguka chini."),   # Bantu noun-class agreement test
]

print("POST-CPT SMOKE TEST")
print("=" * 65)
for en, ref in TEST_SENTENCES:
    hyp = translate(en)
    print(f"EN : {en}")
    print(f"REF: {ref}")
    print(f"HYP: {hyp}")
    print()

## 15. Save all outputs and summary

In [ ]:
# Update metrics with fragmentation results
metrics.update({
    "pre_cpt_fragmentation"  : round(pre_cpt_ratio, 4),
    "post_cpt_fragmentation" : round(post_cpt_ratio, 4),
})

with open(f"{OUTPUT_DIR}/stage1_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("=" * 55)
print("STAGE 1 CPT — COMPLETE")
print("=" * 55)
print(f"Adapter saved to     : {OUTPUT_DIR}/final_adapter/")
print(f"Metrics saved to     : {OUTPUT_DIR}/stage1_metrics.json")
print()
print("What to download from /kaggle/working/:")
print("  stage1_cpt_adapter/final_adapter/   — LoRA weights for Stage 2")
print("  stage1_cpt_adapter/stage1_metrics.json — training summary")
print()
print("What to check before moving to Stage 2 (SFT):")
print("  1. Smoke test outputs should be in Swahili (not Finnish!)")
print("  2. Noun-class agreement: 'Viti vizuri vilianguka' — all vi- prefixes must match")
print("  3. Train loss should be converging (check training logs above)")